# nomono3d · Colab GPU server

**nomono3d** turns flat 360° panoramas and ordinary photos into explorable 3D in the browser: an AI depth map rebuilds the image as real geometry, and you get parallax on a phone (tilt), a desktop (mouse) or with head tracking, plus 3D objects placed on real depth inside panoramas.

This notebook downloads the project from GitHub, starts its server on a free Colab GPU and gives you a **Colab link** to it. The link opens on any device signed in to the same Google account: a computer, a phone or a VR headset. A public link through Cloudflare or another tunnel is optional (cell 3).

| Images | Depth model |
|---|---|
| 360° equirectangular panoramas (2:1, or 1:1 / 4:1 stereo) | [DA360](https://github.com/Insta360-Research-Team/DA360) |
| Ordinary photos, any aspect ratio | [Depth Anything V2](https://github.com/DepthAnything/Depth-Anything-V2) |

**How to run**
1. *Runtime ▸ Change runtime type ▸* **T4 GPU** (any GPU works; CPU works too, slowly).
2. Run cell **1** (install, about 1–2 minutes), then cell **2** (start). Cell 2 prints the Colab link and a QR code for your phone or headset.
3. Open the link, pick a panorama or a photo and press **View**. Depth maps are generated automatically on the GPU.

Source code and documentation: [github.com/yevheniikhan/nomono3d](https://github.com/yevheniikhan/nomono3d)

In [ ]:
#@title 1 · Install { display-mode: "form" }
REPO = "https://github.com/yevheniikhan/nomono3d"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
#@markdown **Depth models** (`large` = best quality, `small` = fastest; `none` switches a model off)
PANO_MODEL = "large"  #@param ["small", "base", "large", "none"]
PHOTO_MODEL = "large"  #@param ["small", "base", "large", "none"]
#@markdown Depth Anything V2 *Base* and *Large* weights are licensed CC-BY-NC-4.0 (non-commercial); *Small* is Apache-2.0. DA360 is MIT.

#@markdown **AI background** for photos: what appears behind objects when the viewpoint moves ([MI-GAN](https://github.com/Picsart-AI-Research/MI-GAN), MIT, ~27 MB). The site lets you switch between it and the built-in fill.
BACKGROUND_MODEL = "migan"  #@param ["migan", "none"]

#@markdown **Library** — where uploaded images and generated depth maps are kept
LIBRARY = "Colab (temporary)"  #@param ["Colab (temporary)", "Google Drive"]
DRIVE_FOLDER = "nomono3d"  #@param {type:"string"}
ALLOW_UPLOAD = True  #@param {type:"boolean"}

import os, subprocess, sys, shutil, time

APP = "/content/nomono3d"
def sh(cmd, check=True):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if check and r.returncode:
        print(r.stdout[-2000:], r.stderr[-2000:]); raise RuntimeError(cmd)
    return r.stdout

gpu = sh("nvidia-smi --query-gpu=name,memory.total --format=csv,noheader", check=False).strip()
print("GPU:", gpu or "none: depth estimation will run on the CPU, slowly. Runtime ▸ Change runtime type ▸ T4 GPU")
if not gpu and "large" in (PANO_MODEL, PHOTO_MODEL):
    print("Tip: without a GPU choose the 'small' models, otherwise a depth map can take several minutes.")

if os.path.isdir(APP + "/.git"):
    sh(f"git -C {APP} fetch -q --depth 1 origin {BRANCH} && git -C {APP} checkout -q -B {BRANCH} FETCH_HEAD")
else:
    sh(f"git clone -q --depth 1 -b {BRANCH} {REPO} {APP}")
print("code:", sh(f"git -C {APP} log -1 --format='%h %s'").strip())

sh(f"{sys.executable} -m pip install -q -r {APP}/server/requirements.txt qrcode")

DATA = None
if LIBRARY == "Google Drive":
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = f"/content/drive/MyDrive/{DRIVE_FOLDER.strip('/') or 'nomono3d'}"
    os.makedirs(DATA, exist_ok=True)
    print("library:", DATA)
else:
    DATA = "/content/nomono3d_library"
    print("library:", DATA, "(cleared when the runtime ends)")
print("\nInstalled. Run cell 2.")

In [ ]:
#@title 2 · Start the server and get the link { display-mode: "form" }
#@markdown The main link is the **Colab link**: it opens on any device signed in to the same Google account (computer, phone, VR headset browser). Nothing else is needed.

#@markdown Optional: also try a **public link** (Cloudflare, ngrok, Pinggy, localhost.run, Gradio) for devices without your Google account. You can do this later in cell 3.
PUBLIC_LINK = False  #@param {type:"boolean"}
TUNNEL = "auto"  #@param ["auto", "Cloudflare named tunnel", "Cloudflare quick tunnel", "ngrok", "Pinggy", "localhost.run", "Gradio share"]
import os, re, subprocess, sys, time, json, urllib.request, io, base64, shutil, secrets as _secrets
from IPython.display import display, HTML

PORT, HEALTH = 8000, "/api/health"
CF = "/content/cloudflared"
LOG = "/content/tunnels"; os.makedirs(LOG, exist_ok=True)
_closers = globals().get("_closers", {})          # provider → function that stops it (kept across reruns)

def _secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return os.environ.get(name)

def _json(url, headers=None, timeout=8):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "nomono3d-colab", **(headers or {})})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return r.status, json.loads(r.read() or b"null")
    except Exception:
        return None, None

def _stop(key):
    f = _closers.pop(key, None)
    if f:
        try: f()
        except Exception: pass

def stop_tunnels():
    for k in list(_closers): _stop(k)

def _popen(key, cmd, env=None):
    _stop(key)
    p = subprocess.Popen(cmd, env=env, stdin=subprocess.DEVNULL, stdout=open(f"{LOG}/{key}.log", "w"), stderr=subprocess.STDOUT)
    _closers[key] = lambda: (p.terminate(), p.wait(5))
    return p

def _grep(key, p, pattern, timeout=45):
    for _ in range(timeout):
        m = re.search(pattern, open(f"{LOG}/{key}.log", errors="ignore").read())
        if m: return m.group(0)
        if p.poll() is not None: return None
        time.sleep(1)

def _wait_ready(p, mport, timeout=60):            # cloudflared /ready = an edge connection is registered
    for _ in range(timeout):
        if p.poll() is not None: return False
        if _json(f"http://127.0.0.1:{mport}/ready")[0] == 200: return True
        time.sleep(1)
    return False

def verify(url, tries=20, pause=3):
    """The link counts only once the app answers through it (a new hostname needs a moment in DNS)."""
    for _ in range(tries):
        h = _json(url.rstrip("/") + HEALTH, {"ngrok-skip-browser-warning": "1"})[1]
        if isinstance(h, dict) and h.get("app") == "nomono3d": return True
        time.sleep(pause)
    return False

def _cloudflared():
    """Download cloudflared only when a Cloudflare tunnel is actually requested."""
    if not (os.path.exists(CF) and os.path.getsize(CF) > 10_000_000 and os.access(CF, os.X_OK)):
        subprocess.run(f"wget -q -O {CF}.part https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
                       f"&& chmod +x {CF}.part && mv {CF}.part {CF}", shell=True, check=True)
    return CF

# 1 · Cloudflare named tunnel: a stable https://<CF_HOSTNAME> on your own domain
def cf_named():
    tok, host = _secret("CF_TUNNEL_TOKEN"), _secret("CF_HOSTNAME")
    if not (tok and host): return None
    p = _popen("cf_named", [_cloudflared(), "tunnel", "--no-autoupdate", "--protocol", "http2", "--edge-ip-version", "4",
                            "--metrics", "127.0.0.1:20243", "run"], env=dict(os.environ, TUNNEL_TOKEN=tok))
    return "https://" + host.replace("https://", "").strip("/") if _wait_ready(p, 20243) else None

# 2 · Cloudflare quick tunnel (no account): HTTP/2 first, then QUIC
def cf_quick(protocol):
    cfg = os.path.expanduser("~/.cloudflared/config.yml")
    if os.path.exists(cfg): os.rename(cfg, cfg + ".off")       # quick tunnels refuse to start with a config file
    mport = 20241 if protocol == "http2" else 20242
    p = _popen(f"cf_{protocol}", [_cloudflared(), "tunnel", "--no-autoupdate", "--protocol", protocol, "--edge-ip-version", "4",
                                  "--metrics", f"127.0.0.1:{mport}", "--url", f"http://127.0.0.1:{PORT}"])
    if not _wait_ready(p, mport): return None
    host = (_json(f"http://127.0.0.1:{mport}/quicktunnel")[1] or {}).get("hostname")
    if not host:
        return _grep(f"cf_{protocol}", p, r"https://(?!api\.)[-a-z0-9]+\.trycloudflare\.com", 10)
    return f"https://{host}"

# 3 · ngrok (free account; NGROK_AUTHTOKEN secret)
def ngrok_link():
    tok = _secret("NGROK_AUTHTOKEN")
    if not tok: return None
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyngrok"], check=False)
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = tok
    dom = _secret("NGROK_DOMAIN")
    t = ngrok.connect(str(PORT), "http", **({"domain": dom} if dom else {}))
    _closers["ngrok"] = lambda: ngrok.disconnect(t.public_url)
    return t.public_url.replace("http://", "https://", 1)

def _ssh():
    if not shutil.which("ssh"):
        subprocess.run("apt-get -qq update && apt-get -qq install -y openssh-client", shell=True, capture_output=True)
    return ["ssh", "-T", "-o", "StrictHostKeyChecking=accept-new", "-o", f"UserKnownHostsFile={LOG}/known_hosts",
            "-o", "ServerAliveInterval=30", "-o", "ExitOnForwardFailure=yes"]

# 4 · Pinggy (no account; a free tunnel closes after 60 minutes)
def pinggy():
    p = _popen("pinggy", _ssh() + ["-p", "443", f"-R0:127.0.0.1:{PORT}", "a.pinggy.io"])
    return _grep("pinggy", p, r"https://[-a-z0-9]+\.(?:[a-z]+\.)*pinggy\.(?:link|online|io)")

# 5 · localhost.run (no account; the free address changes every few hours)
def localhost_run():
    p = _popen("lhr", _ssh() + ["-R", f"80:127.0.0.1:{PORT}", "nokey@localhost.run"])
    return _grep("lhr", p, r"https://(?!admin\.|www\.)[-a-z0-9]+\.(?:lhr\.life|localhost\.run)")

# 6 · Gradio share relay (*.gradio.live, 72 hours)
def gradio_share():
    try:
        from gradio.networking import setup_tunnel
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio"], check=False)
        from gradio.networking import setup_tunnel
    import inspect
    n = len(inspect.signature(setup_tunnel).parameters)
    args = ["127.0.0.1", PORT, _secrets.token_urlsafe(32), None, None][:n]
    url = setup_tunnel(*args)
    try:
        from gradio.tunneling import CURRENT_TUNNELS
        _closers["gradio"] = lambda: [t.kill() for t in CURRENT_TUNNELS]
    except Exception: pass
    return "https://" + url.split("://", 1)[-1]

CHAIN = [("Cloudflare named tunnel", "cf_named", cf_named),
         ("Cloudflare quick tunnel", "cf_http2", lambda: cf_quick("http2")),
         ("Cloudflare quick tunnel", "cf_quic", lambda: cf_quick("quic")),
         ("ngrok", "ngrok", ngrok_link),
         ("Pinggy", "pinggy", pinggy),
         ("localhost.run", "lhr", localhost_run),
         ("Gradio share", "gradio", gradio_share)]

def start_public_link(choice="auto", skip=()):
    """Try the providers in order; return (name, url) of the first link that works end to end."""
    stop_tunnels()
    if choice == "Colab link only": return None, None
    for name, key, fn in CHAIN:
        if (choice != "auto" and name != choice) or key in skip: continue
        print(f"{name}{' (QUIC)' if key == 'cf_quic' else ''} …", end=" ", flush=True)
        try:
            url = fn()
        except Exception as e:
            print(f"{type(e).__name__}: {str(e)[:160]}"); url = None; _stop(key); continue
        if url and verify(url):
            print("OK"); return name, url
        print(f"{url} does not answer" if url else "not available")
        _stop(key)
    return None, None

def colab_link():
    try:
        from google.colab.output import eval_js
        return eval_js(f"google.colab.kernel.proxyPort({PORT}, {{'cache': false}})")
    except Exception:
        return None

def _qr(url):
    try:
        import qrcode
        buf = io.BytesIO(); qrcode.make(url, box_size=5, border=2).save(buf, format="PNG")
        return f'<img alt="QR" style="image-rendering:pixelated;background:#fff;padding:6px;border-radius:6px;width:170px" src="data:image/png;base64,{base64.b64encode(buf.getvalue()).decode()}">'
    except Exception:
        return ""

def show_links(local, public=None, provider=None, tried=False):
    """The Colab link first (with its QR code); a public link only if one was requested and works."""
    card = lambda qr, body: f'<div style="display:flex;gap:20px;align-items:center;padding:14px 16px;border:1px solid #888;border-radius:10px;margin:8px 0">{qr}<div>{body}</div></div>'
    out = '<div style="font-family:system-ui,sans-serif;max-width:860px"><div style="font-weight:600;font-size:15px;margin-bottom:4px">nomono3d is running</div>'
    if local:
        out += card(_qr(local), f"""<div style="font-size:12px;opacity:.7">Colab link · main</div>
          <a href="{local}" target="_blank" style="font-size:17px;word-break:break-all">{local}</a>
          <div style="font-size:12px;opacity:.75;line-height:1.6;margin-top:8px">Opens in any browser signed in to <b>this Google account</b>: here, on your phone (scan the code)
          or in the VR headset browser after signing in to Google there. The <b>Headset</b> button in the viewer builds its links and QR codes from this address.</div>""")
    else:
        out += '<div style="color:#b5452f;margin:8px 0">No Colab link here (is this running outside Colab?). Run cell 3 for a public link.</div>'
    if public:
        out += card(_qr(public), f"""<div style="font-size:12px;opacity:.7">Public link ({provider}) · optional</div>
          <a href="{public}" target="_blank" style="font-size:15px">{public}</a>
          <div style="font-size:12px;opacity:.75;line-height:1.6;margin-top:8px">Works without a Google account, unless your network blocks it.</div>""")
    elif tried:
        out += '<div style="font-size:12px;opacity:.75;margin:6px 0">No public link came up (blocked or unavailable). The Colab link above is enough; see the notes at the end.</div>'
    out += '<div style="font-size:12px;opacity:.6;margin-top:6px">Keep this tab open: the links stop working when the runtime ends.</div></div>'
    display(HTML(out))

# ── server ──
stop_tunnels()
p = globals().get("_n3d_server")
if p and p.poll() is None: p.terminate()
subprocess.run(f"fuser -k {PORT}/tcp", shell=True, capture_output=True)   # a server left over from an earlier run
time.sleep(1)
cmd = [sys.executable, "-u", "-m", "nomono3d_server", "--port", str(PORT), "--data", DATA,
       "--pano-model", PANO_MODEL, "--photo-model", PHOTO_MODEL, "--inpaint", globals().get("BACKGROUND_MODEL", "migan"),
       "--cache", "/content/cache", "--preload"]
if not ALLOW_UPLOAD:
    cmd.append("--no-upload")
env = dict(os.environ, PYTHONUNBUFFERED="1", PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
           PYTHONPATH=os.pathsep.join(p for p in (f"{APP}/server", os.environ.get("PYTHONPATH", "")) if p))
_n3d_server = subprocess.Popen(cmd, cwd=APP, env=env, stdout=open("/content/server.log", "w"), stderr=subprocess.STDOUT)
h = None
for _ in range(120):
    h = _json(f"http://127.0.0.1:{PORT}{HEALTH}", timeout=3)[1]
    if h or _n3d_server.poll() is not None: break
    time.sleep(1)
if not h or _n3d_server.poll() is not None:
    print(open("/content/server.log").read()[-3000:]); raise RuntimeError("the server did not start")
print(f"server {h['version']} · {h['device']} · panoramas: {h['depth']['pano']} · photos: {h['depth']['photo']} · background: {h.get('inpaint')}")

# ── links ──
COLAB_URL = colab_link()
PROVIDER, PUBLIC_URL = start_public_link(TUNNEL) if PUBLIC_LINK else (None, None)
show_links(COLAB_URL, PUBLIC_URL, PROVIDER, tried=PUBLIC_LINK)


In [ ]:
#@title 3 · Optional: public link (Cloudflare, ngrok, Pinggy, localhost.run, Gradio) { display-mode: "form" }
#@markdown Only needed for devices that are not signed in to your Google account. `auto` tries the providers in order; `next` skips the one running now; `off` stops the public link.
TUNNEL3 = "auto"  #@param ["auto", "next", "off", "Cloudflare named tunnel", "Cloudflare quick tunnel", "ngrok", "Pinggy", "localhost.run", "Gradio share"]
if TUNNEL3 == "off":
    stop_tunnels(); PROVIDER, PUBLIC_URL = None, None
else:
    _skip = [k for n, k, _ in CHAIN if n == globals().get("PROVIDER")] if TUNNEL3 == "next" else []
    PROVIDER, PUBLIC_URL = start_public_link("auto" if TUNNEL3 == "next" else TUNNEL3, _skip)
show_links(globals().get("COLAB_URL") or colab_link(), PUBLIC_URL, PROVIDER, tried=TUNNEL3 != "off")


In [ ]:
#@title 4 · Status and logs { display-mode: "form" }
import json, urllib.request, glob
try:
    print(json.dumps(json.load(urllib.request.urlopen("http://127.0.0.1:8000/api/health", timeout=3)), indent=1))
except Exception as e:
    print("server not reachable:", e)
print("Colab link:", globals().get("COLAB_URL"))
if globals().get("PUBLIC_URL"):
    ok = verify(PUBLIC_URL, tries=1)
    print(("public link OK: " if ok else "public link NOT reachable from Colab: ") + f"{PUBLIC_URL} ({PROVIDER})")
print("\n── server.log ──"); print(open("/content/server.log").read()[-4000:])
for f in sorted(glob.glob("/content/tunnels/*.log")):
    lines = [l for l in open(f, errors="ignore").read().splitlines() if "precheck" not in l]
    print(f"── {os.path.basename(f)} ──"); print("\n".join(lines[-15:]))


In [ ]:
#@title Open inside this notebook (if the Colab link shows an error in a new tab) { display-mode: "form" }
HEIGHT = 760  #@param {type:"integer"}
from google.colab import output
output.serve_kernel_port_as_iframe(8000, height=HEIGHT)


In [ ]:
#@title 5 · Stop { display-mode: "form" }
stop_tunnels()
p = globals().get("_n3d_server")
if p and p.poll() is None:
    p.terminate()
print("stopped")


### Notes
- **Colab link**: the address is private to your Google account. On a phone it opens in a browser where you are signed in to the same account (Chrome on Android usually is). In the Meta Quest Browser sign in to google.com once, then scan the QR code or open the link. If a new tab shows *403* or an empty page, open the link again from this notebook (it signs itself in through Colab) or use the *Open inside this notebook* cell. The link changes with every new runtime.
- **Why a public tunnel may not open**: `trycloudflare.com` quick tunnels are widely abused to spread malware, so many antivirus programs, corporate networks and DNS filters block the whole domain. The notebook checks a public link from Colab before showing it, but it cannot see your network's filter. If you need a public link, try another provider in cell 3 (`next`), or use a Cloudflare tunnel on your own domain (below), which filters do not block.
- **A stable public address (optional)**: create a free Cloudflare account with a domain, then *Zero Trust ▸ Networks ▸ Tunnels ▸ Create a tunnel* (type *Cloudflared*), add a public hostname such as `nomono3d.example.com` pointing to `http://localhost:8000`, and copy the tunnel token. In Colab open 🔑 *Secrets* and add `CF_TUNNEL_TOKEN` (the token) and `CF_HOSTNAME` (the hostname), with notebook access on, then run cell 3.
- **Other providers**: `ngrok` needs a free account (add the `NGROK_AUTHTOKEN` secret; `NGROK_DOMAIN` gives a free static address). `Pinggy` and `localhost.run` need no account but their free links expire (60 minutes / a few hours). `Gradio share` lasts 72 hours.
- **VR headset**: open the Colab link in the Meta Quest Browser (signed in to Google), or use **Headset** in the viewer to get a link and QR code for exactly what you set up, 3D objects included.
- **Phones**: every link is `https://`, so device tilt and the front camera (head tracking) work. iOS asks for motion access when you press **View**.
- **Library**: images you save with **⇪ library** and every depth map generated for a library item are stored in the library folder. Choose *Google Drive* in cell 1 to keep them between sessions; the folder then holds `panoramas/`, `photos/` and `models/` and can be copied to any web hosting as-is.
- **Your own hosting**: the web app in `web/` is static. Put it on any HTTPS host (GitHub Pages works) together with the depth maps made here; see the README.
- **Quality**: depth maps are 8-bit PNG, white = near. Panoramas get up to 4096×2048, photos up to 2048 px on the long side.
- **API**: the server documents itself at `/api/docs` (POST `/api/depth` with an image and `kind=pano|photo` returns a PNG).